# QLoRA Adapter Training

Train per-artist LoRA/DoRA adapters on Gemma 4 E4B.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="google/gemma-4-E4B",
    local_dir="./models/gemma-4-E4B",
)

In [ ]:
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

model_path = "./models/gemma-4-E4B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

model = prepare_model_for_kbit_training(model)

train_df = pd.read_csv("./data/train.csv")
train_df["text"] = train_df.apply(lambda row: f"Write song lyrics.\n\n{row['clean']}", axis=1)

In [ ]:
def train_adapter(artist, r=8, use_dora=False, epochs=3, lr=2e-4):
    artist_df = train_df[train_df["artist"] == artist].reset_index(drop=True)
    dataset = Dataset.from_pandas(artist_df[["text"]])

    tag = "dora" if use_dora else "lora"
    output_dir = f"./adapters/{artist.lower().replace(' ', '_')}_{tag}_r{r}"

    lora_config = LoraConfig(
        r=r,
        lora_alpha=r * 2,
        target_modules=r"model\.language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)",
        lora_dropout=0.1,
        bias="none",
        task_type="CAUSAL_LM",
        use_dora=use_dora,
    )

    peft_model = get_peft_model(model, lora_config)

    training_args = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        learning_rate=lr,
        max_length=512,
        bf16=True,
        logging_steps=5,
        save_strategy="epoch",
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        gradient_checkpointing=True,
        report_to="none",
        weight_decay=0.05,
    )

    trainer = SFTTrainer(
        model=peft_model,
        train_dataset=dataset,
        args=training_args,
        processing_class=tokenizer,
    )

    trainer.train()
    peft_model.save_pretrained(output_dir)
    peft_model.unload()
    print(f"Saved: {output_dir} ({len(artist_df)} songs, {tag}, r={r})")
    return output_dir

## Main adapters

In [ ]:
train_adapter("Gojira", r=8, use_dora=False)
train_adapter("Tool", r=8, use_dora=True)

## Ablation: LoRA vs DoRA (same artist)

In [ ]:
train_adapter("Gojira", r=8, use_dora=True)
train_adapter("Tool", r=8, use_dora=False)

## Ablation: Rank

In [ ]:
train_adapter("Gojira", r=4, use_dora=False)
train_adapter("Gojira", r=16, use_dora=False)